# Huấn luyện SentencePiece Tokenizer cho Tóm tắt Tiếng Việt (SentencePiece Vietnamese Tokenizer Training)

Notebook này được thiết kế để chạy trên Kaggle nhằm huấn luyện một mô hình **SentencePiece Tokenizer (mã hóa văn bản)** chất lượng cao cho tiếng Việt, làm cơ sở để xây dựng và huấn luyện mô hình học sâu Transformer.

### Các bước thực hiện chính:
1. **Chuẩn bị dữ liệu**: Đọc file dữ liệu huấn luyện sạch `train.jsonl` từ bước làm sạch trước và chuyển đổi thành file văn bản cơ sở `tokenizer_corpus.txt` với cấu trúc xen kẽ dòng `Contents` và `Summary`.
2. **Phân chia tập thử nghiệm**: Trích xuất **20% dữ liệu** làm tập huấn luyện thử và **10% dữ liệu** làm tập đánh giá kiểm thử.
3. **Huấn luyện thử nghiệm**: Chạy thử nghiệm SentencePiece với kiểu mô hình `unigram` trên 3 kích thước từ điển khác nhau: `[16000, 32064, 64064]`.
4. **Đánh giá & Lựa chọn**: Tính toán chỉ số **Fertility Rate (Tỷ lệ mã hóa)** trên tập kiểm thử để chọn ra kích thước từ điển (vocab size) tối ưu nhất cho tiếng Việt.
5. **Huấn luyện thực tế**: Tiến hành huấn luyện thật mô hình SentencePiece Tokenizer trên **toàn bộ** dữ liệu `tokenizer_corpus.txt` với kích thước từ điển tốt nhất đã lựa chọn.
6. **Thử nghiệm mã hóa**: Trực quan hóa kết quả giải mã (decode) và phân tách token thực tế của mô hình.

In [1]:
# Cài đặt thư viện sentencepiece nếu môi trường chưa có
!pip install -q sentencepiece

import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sentencepiece as spm

# Cấu hình style biểu đồ
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Cấu hình Đường dẫn Dữ liệu trên Kaggle
Mặc định, file `train.jsonl` sau khi được xuất từ bước làm sạch trước sẽ nằm ở thư mục làm việc của Kaggle (`/kaggle/working/train.jsonl`). Đoạn code dưới đây sẽ tự động kiểm tra và cấu hình đường dẫn chính xác cho bạn.

In [2]:
# Đường dẫn dự kiến trên Kaggle (nếu chạy từ output của notebook trước)
INPUT_JSONL = "/kaggle/input/notebooks/tranducthinh2006/clean-data/train.jsonl"

# Nếu chạy ở máy local hoặc upload lên Kaggle Dataset, quét tìm tự động
if not os.path.exists(INPUT_JSONL):
    found = False
    for dirname, _, filenames in os.walk('/kaggle'):
        for filename in filenames:
            if filename == "train.jsonl":
                INPUT_JSONL = os.path.join(dirname, filename)
                print(f"Tự động tìm thấy file train.jsonl tại: {INPUT_JSONL}")
                found = True
                break
        if found: break
    if not found:
        # Đường dẫn tương đối chạy local
        INPUT_JSONL = "../data/processed/train.jsonl"
        print(f"Chạy chế độ cục bộ local, sử dụng đường dẫn: {INPUT_JSONL}")
else:
    print(f"Tìm thấy file train.jsonl tại thư mục gốc Kaggle: {INPUT_JSONL}")

OUTPUT_DIR = "/kaggle/working/data/processed/tokenizer" if "/kaggle" in INPUT_JSONL else "../data/processed/tokenizer"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Thư mục làm việc đầu ra: {OUTPUT_DIR}")

Tìm thấy file train.jsonl tại thư mục gốc Kaggle: /kaggle/input/notebooks/tranducthinh2006/clean-data/train.jsonl
Thư mục làm việc đầu ra: /kaggle/working/data/processed/tokenizer


## 2. Tạo File Cơ sở Dữ liệu Tokenizer Corpus (`tokenizer_corpus.txt`)
Đọc từ file dữ liệu huấn luyện `train.jsonl` và chuyển đổi sang dạng xen kẽ:
```text
Contents dòng 1
Summary dòng 1
Contents dòng 2
Summary dòng 2
...
```
Cấu trúc này giúp SentencePiece học phân bổ token đồng đều của cả phần nội dung chi tiết lẫn phần tóm tắt ngắn.

In [3]:
corpus_path = os.path.join(OUTPUT_DIR, "tokenizer_corpus.txt")
pairs = []

print(f"Đang đọc dữ liệu từ: {INPUT_JSONL}")
with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            item = json.loads(line)
            c = item.get("Contents", "").strip()
            s = item.get("Summary", "").strip()
            if c and s:
                pairs.append((c, s))

print(f"Đọc thành công {len(pairs)} dòng cặp dữ liệu.")

# Ghi ra file text dạng xen kẽ
print(f"Đang ghi dữ liệu xen kẽ ra file: {corpus_path}")
with open(corpus_path, "w", encoding="utf-8") as f:
    for c, s in pairs:
        f.write(c + "\n")
        f.write(s + "\n")

print("Đã lưu thành công file tokenizer_corpus.txt!")

Đang đọc dữ liệu từ: /kaggle/input/notebooks/tranducthinh2006/clean-data/train.jsonl
Đọc thành công 197580 dòng cặp dữ liệu.
Đang ghi dữ liệu xen kẽ ra file: /kaggle/working/data/processed/tokenizer/tokenizer_corpus.txt
Đã lưu thành công file tokenizer_corpus.txt!


## 3. Trích xuất Dữ liệu Đánh giá & Huấn luyện Thử nghiệm
Chúng ta tiến hành trích xuất ngẫu nhiên dữ liệu với seed cố định (`random_state=42`):
- **20% dữ liệu** làm tập Train thử (`tuning_train.txt`).
- **10% dữ liệu** làm tập Test thử (`tuning_test.txt`).

Việc trích xuất này được thực hiện ở cấp độ **Cặp (Contents, Summary)** để đảm bảo mối liên kết chặt chẽ.

In [4]:
# Thiết lập seed để đảm bảo tính tái lập
random.seed(42)
random.shuffle(pairs)

num_samples = len(pairs)
num_train_tuning = int(num_samples * 0.2)
num_test_tuning = int(num_samples * 0.1)

train_tuning_pairs = pairs[:num_train_tuning]
test_tuning_pairs = pairs[num_train_tuning : num_train_tuning + num_test_tuning]

print(f"Kích thước tập Train thử (20%): {len(train_tuning_pairs)} mẫu")
print(f"Kích thước tập Test thử (10%): {len(test_tuning_pairs)} mẫu")

# Lưu file train thử
tuning_train_path = os.path.join(OUTPUT_DIR, "tuning_train.txt")
with open(tuning_train_path, "w", encoding="utf-8") as f:
    for c, s in train_tuning_pairs:
        f.write(c + "\n")
        f.write(s + "\n")
print(f"Đã xuất file train thử: {tuning_train_path}")

# Lưu file test thử (phẳng hóa danh sách các câu)
test_sentences = []
for c, s in test_tuning_pairs:
    test_sentences.append(c)
    test_sentences.append(s)

tuning_test_path = os.path.join(OUTPUT_DIR, "tuning_test.txt")
with open(tuning_test_path, "w", encoding="utf-8") as f:
    for sent in test_sentences:
        f.write(sent + "\n")
print(f"Đã xuất file test thử: {tuning_test_path}")

Kích thước tập Train thử (20%): 39516 mẫu
Kích thước tập Test thử (10%): 19758 mẫu
Đã xuất file train thử: /kaggle/working/data/processed/tokenizer/tuning_train.txt
Đã xuất file test thử: /kaggle/working/data/processed/tokenizer/tuning_test.txt


## 4. Huấn luyện Thử nghiệm & Đánh giá Fertility Rate
Tiến hành huấn luyện thử các Candidate Vocab Size: **`[16000, 32064, 64064]`** bằng thuật toán **SentencePiece Unigram** với cấu hình đặc biệt:
- `character_coverage = 0.9995`
- Special Tokens:
  - `<pad>` = 0
  - `<unk>` = 1
  - `<bos>` = 2
  - `<eos>` = 3

### Khái niệm Fertility Rate (Tỷ lệ mã hóa):
$$\text{Fertility Rate} = \frac{\text{Tổng số lượng Token sinh ra sau mã hóa}}{\text{Tổng số lượng từ thực tế trong câu gốc}}$$
- Tỷ lệ Fertility Rate lý tưởng cho tiếng Việt thường nằm trong khoảng **0.85 đến 1.15**.
- Tỷ lệ quá cao (như > 1.5) biểu thị từ vựng quá nhỏ, từ bị cắt nát vụn (Over-fragmentation) làm tăng chiều dài chuỗi đầu vào Transformer gây chậm/lỗi.
- Tỷ lệ quá gần 1.0 (như 1.05) biểu thị từ vựng quá lớn, chứa nhiều từ hiếm (Over-fitting) làm mô hình học kém đi.

In [5]:
import os
import sentencepiece as spm

def evaluate_fertility(model_prefix, sentences):
    """Tính toán Fertility Rate trên danh sách câu (Tối ưu tốc độ)"""
    sp = spm.SentencePieceProcessor(model_file=f"{model_prefix}.model")
    
    # Đếm tổng số từ gốc (âm tiết) bằng cách join toàn bộ text rồi split, 
    # cực kỳ nhanh trong Python so với việc lặp từng câu
    total_words = len(" ".join(sentences).split())
    if total_words == 0: return 0
    
    # Sử dụng encode_as_ids cho list thay vì lặp từng phần tử (Batch encoding)
    encoded_sentences = sp.encode_as_ids(sentences)
    total_tokens = sum(len(tokens) for tokens in encoded_sentences)
        
    return total_tokens / total_words

vocab_candidates = [16000, 32064] # Tôi khuyên bạn tạm bỏ 64064 ra để test cho nhanh
results = {}

print("--- BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN THỬ NGHIỆM CORPUS ---")
for vocab in vocab_candidates:
    temp_prefix = os.path.join(OUTPUT_DIR, f"spm_temp_{vocab}")
    print(f"Đang huấn luyện thử nghiệm với Vocab Size = {vocab}...")
    
    spm.SentencePieceTrainer.train(
        input=tuning_train_path,
        model_prefix=temp_prefix,
        vocab_size=vocab,
        model_type="unigram",
        character_coverage=0.9995,
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
        pad_piece="<pad>",
        unk_piece="<unk>",
        bos_piece="<bos>",
        eos_piece="<eos>",
        # --- CÁC THAM SỐ BỔ SUNG QUAN TRỌNG ---
        byte_fallback=True
    )
    
    # Đánh giá Fertility Rate
    fertility = evaluate_fertility(temp_prefix, test_sentences)
    results[vocab] = fertility
    print(f"  => Hoàn thành! Chỉ số Fertility Rate trên tập Test thử: {fertility:.4f}\n")
    
    # Dọn dẹp file tạm
    if os.path.exists(f"{temp_prefix}.model"): os.remove(f"{temp_prefix}.model")
    if os.path.exists(f"{temp_prefix}.vocab"): os.remove(f"{temp_prefix}.vocab")

--- BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN THỬ NGHIỆM CORPUS ---
Đang huấn luyện thử nghiệm với Vocab Size = 16000...


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /kaggle/working/data/processed/tokenizer/tuning_train.txt
  input_format: 
  model_prefix: /kaggle/working/data/processed/tokenizer/spm_temp_16000
  model_type: UNIGRAM
  vocab_size: 16000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <bos>
  

  => Hoàn thành! Chỉ số Fertility Rate trên tập Test thử: 1.1518

Đang huấn luyện thử nghiệm với Vocab Size = 32064...
  => Hoàn thành! Chỉ số Fertility Rate trên tập Test thử: 1.1428



e.cc(427) LOG(INFO) Adding meta_piece: <0xAE>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <0xAF>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <0xB0>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <0xB1>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <0xB2>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <0xB3>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <0xB4>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <0xB5>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <0xB6>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <0xB7>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <0xB8>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <0xB9>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <0xBA>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <0xBB>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <0xBC>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <0xBD>
trainer_interface.cc(427

## 6. Huấn luyện Thật trên Toàn bộ Corpus (`tokenizer_corpus.txt`)
Sử dụng kích thước từ điển tốt nhất đã lựa chọn ở trên để tiến hành huấn luyện thật sự trên toàn bộ dữ liệu. Kết quả lưu trữ sẽ gồm 2 file `.model` và `.vocab` chuẩn dùng cho mô hình Transformer.

In [6]:
# Đường dẫn lưu mô hình thật
final_output_dir = "/kaggle/working/data/tokenizer" if "/kaggle" in INPUT_JSONL else "../data/tokenizer"
os.makedirs(final_output_dir, exist_ok=True)
final_prefix = os.path.join(final_output_dir, "vietnamese_spm")

print(f"=== BẮT ĐẦU HUẤN LUYỆN TOÀN BỘ DỮ LIỆU CORPUS VỚI VOCAB SIZE = {16000} ===")
spm.SentencePieceTrainer.train(
    input=corpus_path,
    model_prefix=final_prefix,
    vocab_size=16000,
    model_type="unigram",
    character_coverage=0.9995,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    pad_piece="<pad>",
    unk_piece="<unk>",
    bos_piece="<bos>",
    eos_piece="<eos>"
)

print("\nHuấn luyện Tokenizer thật sự đã hoàn thành thành công!")
print(f"  - File Model: {final_prefix}.model")
print(f"  - File Vocab: {final_prefix}.vocab")

=== BẮT ĐẦU HUẤN LUYỆN TOÀN BỘ DỮ LIỆU CORPUS VỚI VOCAB SIZE = 16000 ===

Huấn luyện Tokenizer thật sự đã hoàn thành thành công!
  - File Model: /kaggle/working/data/tokenizer/vietnamese_spm.model
  - File Vocab: /kaggle/working/data/tokenizer/vietnamese_spm.vocab


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /kaggle/working/data/processed/tokenizer/tokenizer_corpus.txt
  input_format: 
  model_prefix: /kaggle/working/data/tokenizer/vietnamese_spm
  model_type: UNIGRAM
  vocab_size: 16000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <bos>
  eos_pi

## 7. Trực quan Thử nghiệm Tokenizer đã Huấn luyện
Chúng ta nạp mô hình vừa huấn luyện xong và mã hóa thử một số câu văn tiếng Việt ngẫu nhiên để quan sát thực tế.

In [7]:
# Nạp mô hình SentencePiece vừa huấn luyện
sp = spm.SentencePieceProcessor(model_file=f"{final_prefix}.model")

test_text = "Mô hình Transformer được sử dụng rất phổ biến trong bài toán tóm tắt văn bản tiếng Việt nhờ khả năng nắm bắt ngữ cảnh tốt."

print(f"Văn bản gốc:\n\"{test_text}\"\n")

# Mã hóa thành danh sách Subwords
tokens = sp.encode_as_pieces(test_text)
print("Danh sách Subwords (Token):")
print(tokens)
print(f"  -> Số lượng Token: {len(tokens)}")
print(f"  -> Số lượng từ khoảng trắng gốc: {len(test_text.split())}")
print(f"  -> Fertility Rate của câu thử nghiệm: {len(tokens)/len(test_text.split()):.3f}\n")

# Mã hóa thành ID số để train mô hình
ids = sp.encode_as_ids(test_text)
print("Danh sách ID số truyền vào Transformer:")
print(ids)

# Giải mã ngược lại
decoded = sp.decode_pieces(tokens)
print(f"\nGiải mã ngược lại chuỗi gốc:\n\"{decoded}\"")

Văn bản gốc:
"Mô hình Transformer được sử dụng rất phổ biến trong bài toán tóm tắt văn bản tiếng Việt nhờ khả năng nắm bắt ngữ cảnh tốt."

Danh sách Subwords (Token):
['▁Mô', '▁hình', '▁Trans', 'form', 'er', '▁được', '▁sử', '▁dụng', '▁rất', '▁phổ', '▁biến', '▁trong', '▁bài', '▁toán', '▁tóm', '▁tắt', '▁văn', '▁bản', '▁tiếng', '▁Việt', '▁nhờ', '▁khả', '▁năng', '▁nắm', '▁bắt', '▁ngữ', '▁cảnh', '▁tốt', '.']
  -> Số lượng Token: 29
  -> Số lượng từ khoảng trắng gốc: 26
  -> Fertility Rate của câu thử nghiệm: 1.115

Danh sách ID số truyền vào Transformer:
[5097, 147, 10967, 11397, 1957, 12, 208, 114, 191, 1058, 391, 14, 687, 837, 5022, 2240, 359, 262, 634, 59, 1253, 490, 155, 1168, 335, 1671, 301, 309, 5]

Giải mã ngược lại chuỗi gốc:
"Mô hình Transformer được sử dụng rất phổ biến trong bài toán tóm tắt văn bản tiếng Việt nhờ khả năng nắm bắt ngữ cảnh tốt."
